In [2]:
from pathlib import Path
import pyvista as pv
import numpy as np
import pandas as pd
import gdist
from scipy.spatial.distance import cdist
import json
from tqdm import tqdm

from phd_helpers.MeshQuality import (
        compute_d_metrics, compute_dists, compute_rmsd, sample_surface,
    )
from phd_helpers.paths import get_mesh, get_subject_stl_path, get_boundary

# Assign cell/point arrays to surfaces and volumes
---
 - Bone_orig inner
 - Bone_smooth inner
 - Bone_remesh2d inner
 - Bone_remesh3d inner  
---
 - Cartilage_orig inner
 - Cartilage_smooth inner
 - Cartilage_remesh2d inner
 - Cartilage_remesh3d inner
---
 - Cartilage_tet inner
---

In [2]:
def build_ids(ids: list):
    id_1, id_2, id_3 = ids

    id_2d = f'-{id_1}'
    id_cart = f'-{id_1}-{id_2}'
    id_3d = f'-{id_1}-{id_2}-{id_3}'
    return id_2d, id_cart, id_3d

def assign_inner(mesh, mesh_with_inner):
    mesh['inner_cells'] = mesh_with_inner['inner_cells'][mesh_with_inner.find_closest_cell(mesh.cell_centers().points)]
    mesh['inner_points'] = np.full(mesh.n_points, 1)
    mesh['inner_points'][np.unique(mesh.faces.reshape(-1, 4)[:, 1:][mesh['inner_cells']==0])] = 0

def generate_mesh_dict(path_mesh, ids, bone, stl_path, taper_width):
    id_2d, id_cart, id_3d = build_ids(ids)
    
    path2d = path_mesh / '2Dmesh'
    path3d = path_mesh / '3Dmesh'

    # -------- COMBINED --------------------------------------------- #
    mesh2d = pv.read(path2d / f'bone_cartilage_mesh{id_cart}.vtp')

    mesh3d = pv.read(path3d / f'mesh{id_3d}.vtu')
    mesh3d['mesh3d_point_id'] = np.arange(mesh3d.n_points)
    shell_mesh = mesh3d.extract_cells_by_type(5)

    # -------- BONE --------------------------------------------- #
    bone_orig = get_mesh(stl_path, bone) # no inner cells
    bone_smooth = pv.read(path2d / f'bone_smooth{id_2d}.obj') # no inner cells
    bone_remesh2d = mesh2d.extract_cells(mesh2d['region_id']==2, invert=True).extract_surface(algorithm=None) # no inner cells
    bone_remesh3d = shell_mesh.extract_cells(shell_mesh['region_id']==-2, invert=True).extract_surface(algorithm=None) # no inner cells

    # assign region ids to bone_orig and bone_smooth
    #bone_orig['region_id'] = bone_remesh3d['region_id'][bone_remesh3d.find_closest_cell(bone_orig.cell_centers().points)] * -1
    #bone_smooth['region_id'] = bone_remesh3d['region_id'][bone_remesh3d.find_closest_cell(bone_smooth.cell_centers().points)] * -1

    # compute inner region for bone_remesh2d
    bone_remesh2d['bone_remesh2d_point_id'] = np.arange(bone_remesh2d.n_points)
    bone_remesh2d['bone_remesh2d_cell_id'] = np.arange(bone_remesh2d.n_cells)
    inter_remesh2d = bone_remesh2d.extract_cells(bone_remesh2d['region_id']==3).extract_surface(algorithm=None)
    inter_remesh2d['inter_remesh2d_point_id'] = np.arange(inter_remesh2d.n_points)
    inter_bound2d_ids = get_boundary(inter_remesh2d)['inter_remesh2d_point_id']
    geo_dists = gdist.compute_gdist(
        inter_remesh2d.points.astype(np.float64),
        inter_remesh2d.faces.reshape(-1, 4)[:, 1:].astype(np.int32),
        source_indices=inter_bound2d_ids.astype(np.int32), 
    ) 
    inner_mask = geo_dists > taper_width

    # assign inner cells and points arrays to bone_remesh2d
    bone_remesh2d['inner_points'] = np.zeros(bone_remesh2d.n_points, dtype=int)
    bone_remesh2d['inner_points'][inter_remesh2d['bone_remesh2d_point_id'][inner_mask]] = 1
    bone_remesh2d['inner_cells'] = np.zeros(bone_remesh2d.n_cells, dtype=int)
    bone_remesh2d['inner_cells'][bone_remesh2d.extract_points(bone_remesh2d['inner_points'].astype(bool), adjacent_cells=False)['bone_remesh2d_cell_id']] = 1
    # assign to others
    assign_inner(bone_remesh3d, bone_remesh2d)
    assign_inner(bone_smooth, bone_remesh2d)
    assign_inner(bone_orig, bone_remesh2d)

    # -------- CARTILAGE --------------------------------------------- #
    cart_orig = pv.read(path2d / f'orig_cart_surf{id_cart}.vtp') # inner cells
    cart_smooth = pv.read(path2d / f'smooth_cart_surf{id_cart}.vtp') # inner cells
    cart_remesh2d = mesh2d.extract_cells(mesh2d['region_id']==2) # inner cells
    cart_remesh3d = shell_mesh.extract_cells(shell_mesh['region_id']==-2).extract_surface(algorithm=None) # no inner cells

    # assign inner cells and points arrays to cart_remesh3d
    assign_inner(cart_remesh3d, cart_orig)


    # -------- TETRAHEDRAL --------------------------------------------- #
    cart_tet = mesh3d.extract_cells(mesh3d['region_id']==2)
    bone_tet = mesh3d.extract_cells(mesh3d['region_id']==1)

    # compute inner region of cart_tet #
    # mesh3d ids of points that are part of bone_remesh inner region
    bone_mesh3d_inner_ids = bone_remesh3d['mesh3d_point_id'][np.unique(bone_remesh3d.regular_faces[bone_remesh3d['inner_cells']==1])]
    # mesh3d ids of points that are part of cart_remesh inner region
    cart_mesh3d_inner_ids = cart_remesh3d['mesh3d_point_id'][np.unique(cart_remesh3d.regular_faces[cart_remesh3d['inner_cells']==1])]

    cart_tet_cells = cart_tet['mesh3d_point_id'][cart_tet.cells.reshape(-1, 5)[:, 1:]]
    # mask of cart_tet cells that form the bone_remesh3d inner region faces
    bone_mask = np.isin(cart_tet_cells, bone_mesh3d_inner_ids).sum(axis=1) >= 3
    # mask of cart_tet cells that form the cart_remesh3d inner region faces
    cart_mask = np.isin(cart_tet_cells, cart_mesh3d_inner_ids).sum(axis=1) >= 3

    # get mask of cart_tet cells that are > taper width from the boundary
    boundary_points = inter_remesh2d.points[inter_bound2d_ids]
    d = cdist(cart_tet.cell_centers().points, boundary_points).min(axis=1) # min dist of each tet to boundary 
    tet_mask = d > taper_width
    # get mask of cart_tet cells that don't have a face on the boundary
    cart_shell = cart_tet.extract_surface(algorithm=None)
    cart_shell_mesh3d_ids = cart_shell['mesh3d_point_id'][np.unique(cart_shell.regular_faces)]
    cart_interior_tet_mask = ~(np.isin(cart_tet_cells, cart_shell_mesh3d_ids).sum(axis=1) >= 3)
    # get mask of cart_tet cells that are > taper width from the boundary and don't have a face on the surface
    inner_tet_mask = tet_mask & cart_interior_tet_mask

    # assign inner cells to cart_tet
    cart_tet['inner_cells'] = np.zeros(cart_tet.n_cells, dtype=int)
    cart_tet['inner_cells'][bone_mask | cart_mask | inner_tet_mask] = 1

    # extract inner region
    cart_tet_inner = cart_tet.extract_cells(cart_tet['inner_cells']==1)


    # organise into dictionary

    parts = ['bone', 'cart']

    mesh_dict = {
        'bone':{
            'orig': bone_orig,
            'smooth': bone_smooth,
            'remesh2d': bone_remesh2d,
            'remesh3d': bone_remesh3d
        },
        'cart':{
            'orig': cart_orig,
            'smooth': cart_smooth,
            'remesh2d': cart_remesh2d,
            'remesh3d': cart_remesh3d
        }
    }

    for part in parts:
        part_dict = mesh_dict[part]
        for key, mesh in part_dict.items():
            part_dict[key] = {}
            part_dict[key]['full'] = mesh
            part_dict[key]['inner'] = mesh.extract_points(mesh['inner_points']==1, adjacent_cells=False).extract_surface(algorithm=None)
            part_dict[key]['outer'] = mesh.extract_points(mesh['inner_points']!=1, adjacent_cells=False).extract_surface(algorithm=None)

    mesh_dict['tet'] = {
        'bone': bone_tet,
        'cart': cart_tet,
        'cart_inner': cart_tet_inner
    }

    return mesh_dict

# What metrics to compute?

### Cartilage
 - Not sure if to just compare back to orig or just to remesh? 
 - Don't like the idea of talking about both in the write up because it will get complicated.
 - So measure both and decide later

    - inner cart_remesh3d -> cart_remesh2d 
    - inner cart_remesh3d -> cart_smooth
    - inner cart_remesh3d -> cart_orig
    - outer cart_remesh3d -> cart_remesh2d 
    - outer cart_remesh3d -> cart_smooth
    - outer cart_remesh3d -> cart_orig

### Bone
 - Same for bone

    - inner bone_remesh3d -> bone_remesh2d 
    - inner bone_remesh3d -> bone_smooth
    - inner bone_remesh3d -> bone_orig
    - outer bone_remesh3d -> bone_remesh2d 
    - outer bone_remesh3d -> bone_smooth
    - outer bone_remesh3d -> bone_orig

### Element quality & count
- Only need tetrahedral mesh for this

    - Total count
    - Global quality 
    - inner cartilage surface quality 
    - inner cartilge volume quality

### Runtime
 - 3Dmesh step total runtime


## Element Quality
 - min_angle (dihedral) 70.53 is max and ideal
 - radius_ratio 1 is min and ideal
 - aspect ratio 1 is min and ideal
 - scaled_jacobian 1 is max and ideal

In [62]:
METRICS = ['min_angle', 'radius_ratio', 'aspect_ratio', 'scaled_jacobian']
ACCEPTABLE_RANGE = {
    'min_angle': (10, 70.53),
    'radius_ratio': (1.0, 5.0),
    'aspect_ratio': (1.0, 5.0),
    'scaled_jacobian': (0.2, 1.0)
}
PREFERRED_RANGE = {
    'min_angle': (15, 70.53),
    'radius_ratio': (1.0, 3.0),
    'aspect_ratio': (1.0, 3.0),
    'scaled_jacobian': (0.4, 1.0)
}
IDEAL_VALUE = {
    'min_angle': 70.53,
    'radius_ratio': 1.0,
    'aspect_ratio': 1.0,
    'scaled_jacobian': 1.0
}
BAD_SIDE = {
    'min_angle': 'low',
    'radius_ratio': 'high',
    'aspect_ratio': 'high',
    'scaled_jacobian': 'low',
}

def compute_quality_metrics(mesh, dic=None, label=''):
    quality = mesh.cell_quality(METRICS)

    if dic is None:
        dic = {}

    dic |= {
    f"{label}n_cells": mesh.n_cells,
    }

    for metric in METRICS:
        # handle NAN values
        vals = np.asarray(quality[metric], dtype=float)
        finite = np.isfinite(vals)
        n_invalid = np.sum(~finite)
        vals = vals[finite]
        if len(vals) == 0:
            dic |= {
                f"{label}{metric}_n_nan": int(n_invalid),
                f"{label}{metric}_bad_cells": mesh.n_cells,
            }
            continue


        ideal = IDEAL_VALUE[metric]

        # distance from ideal
        dists = np.abs(vals - ideal)
        # closest to ideal
        best_val = vals[np.argmin(dists)]
        # furthest from ideal
        worst_val = vals[np.argmax(dists)]

        # % within preferred range
        vmin_p, vmax_p = PREFERRED_RANGE[metric]
        within_p = np.logical_and(vals >= vmin_p, vals <= vmax_p)
        pct_within_p = 100 * np.sum(within_p) / mesh.n_cells

        # % within acceptable range
        vmin_a, vmax_a = ACCEPTABLE_RANGE[metric]
        within_a = np.logical_and(vals >= vmin_a, vals <= vmax_a)
        pct_within_a = 100 * np.sum(within_a) / mesh.n_cells

        # count outside the acceptable range
        outside_count = (len(vals) - np.sum(within_a)) + n_invalid

        if BAD_SIDE[metric] == 'low':
            # worst below ideal = 5th pct
            pct_95 = np.percentile(vals, 5)
            pct_99 = np.percentile(vals, 1)
            pct_999 = np.percentile(vals, 0.1)
        else:
            # worst above ideal  = 95th percentile
            pct_95 = np.percentile(vals, 95)
            pct_99 = np.percentile(vals, 99)
            pct_999 = np.percentile(vals, 99.9)

# percentages are over all mesh cells
# but summary stats (mean, median, std, percentiles) are over valid cells only
# often nan values so need to keep eye on n_nan count
        data = ({
            f"{label}{metric}_n_nan": int(n_invalid),
            f"{label}{metric}_mean": vals.mean(),
            f"{label}{metric}_median": np.median(vals),
            f"{label}{metric}_std": np.std(vals),
            f"{label}{metric}_best": best_val,
            f"{label}{metric}_worst": worst_val,
            f"{label}{metric}_preferred_range_pct": pct_within_p,
            f"{label}{metric}_acceptable_range_pct": pct_within_a,
            f"{label}{metric}_bad_cells": outside_count,
            f"{label}{metric}_95%": pct_95,
            f"{label}{metric}_99%": pct_99,
            f"{label}{metric}_99.9%": pct_999,
        })

        dic |= data


    return dic


## Distance and Quality main

In [ ]:

# -------- PATHS --------------------------------------------- #
root_dir = Path('../../../../MeshPipeline/outputs/ParamOptimisation/criteria3D') # path to parent of output_root in set_parameters
out_dir = Path('outputs/study1') # path dir to save outputs in

study_prefix = 'study1' # start of dir name of output_root in set_parameters
studies = ['a', 'b', 'c'] # individual study identifier (end of dir name of output_root in set_parameters)

for study in studies:
    study_name = study_prefix + study
    study_dir = root_dir / study_name

    # -------- PARAMS --------------------------------------------- #
    params_path = study_dir / 'params/full_params.json'
    with open(params_path, 'r') as f:
        params = json.load(f)

    taper_width = params['cartilage']['taper_width']
    id_2d, id_cart = 0, 0
    subs = params['subjects']['subject_sideL']
    bone_pairs = params['subjects']['bone_arbone']
    parts = ['bone', 'cart']
    n_samples = 20000

    # -------- MAIN --------------------------------------------- #
    data = {
        'bone': [],
        'cart': [],
        'qual': []
    }
    for sub in subs:

        subject, sideL = sub[:-1], sub[-1]
        stl_path = get_subject_stl_path(subject, sideL)

        for bone_pair in bone_pairs:

            bone, arbone = bone_pair.split('-')
            path_mesh = study_dir / f'meshes/{subject}{sideL}/{bone_pair}'
            run_ids = np.sort([int(p.name.split('-')[-1][:-4]) for p in (path_mesh / '3Dmesh').iterdir() if p.suffix == '.vtu'])

            for run_id in tqdm(run_ids):

                mesh_dict = generate_mesh_dict(path_mesh, [id_2d, id_cart, run_id], bone, stl_path, taper_width)

                for part in parts:
                    
                    # create data row
                    mets = {
                        'sub': sub,
                        'bone': bone,
                        'run_id': str(run_id)+study
                    }
                    
                    mesh = mesh_dict[part]
                    remesh3d = mesh['remesh3d']
                    full_3d = remesh3d['full']
                    inner_3d = remesh3d['inner']
                    outer_3d = remesh3d['outer']

                    # distance from remesh3d
                    for name, regions in mesh.items(): # orig smooth remesh2d remesh3d
                        if name != 'remesh3d':
                            full = regions['full']
                            inner = regions['inner']
                            outer = regions['outer']
                            
                            di_ab = compute_dists(sample_surface(inner_3d, n_samples), full)
                            di_ba = compute_dists(sample_surface(inner, n_samples), full_3d)
                            di = np.hstack( (di_ab, di_ba) )
                        
                            do_ab = compute_dists(sample_surface(outer_3d, n_samples), full)
                            do_ba = compute_dists(sample_surface(outer, n_samples), full_3d)
                            do = np.hstack( (do_ab, do_ba) )

                            label = f'{name}_'
                            mets[label + 'rmsdi'] = compute_rmsd(di)
                            mets = compute_d_metrics(di, mets, label + 'di_')
                            mets[label + 'rmsdo'] = compute_rmsd(do)
                            mets = compute_d_metrics(do, mets, label + 'do_')

                            # height of inner cartilage for: orig smooth remesh2d
                            if part == 'cart':
                                bone_full = mesh_dict['bone']['smooth']['full']
                                h = compute_dists(sample_surface(inner, n_samples), bone_full)
                                mets[label + 'h_rmsd'] = compute_rmsd(h)
                                mets = compute_d_metrics(h, mets, label + 'h_')

                    # height of cartilage for remesh3d
                    if part == 'cart':
                        bone_full = mesh_dict['bone']['remesh3d']['full']
                        h = compute_dists(sample_surface(inner_3d, n_samples), bone_full)
                        mets['remesh3d_h_rmsd'] = compute_rmsd(h)
                        mets = compute_d_metrics(h, mets, 'remesh3d_h_')

                    # only store volume of bone remesh3d
                    if part == 'bone':
                        mets['remesh3d_vol'] = full_3d.volume

                    data[part].append(mets)

                # compute quality of tetrahedral mesh
                row = {
                    'sub': sub,
                    'bone': bone,
                    'run_id': str(run_id)+study
                }
                for name, tet in mesh_dict['tet'].items(): # bone cart cart_inner
                    row = compute_quality_metrics(tet, row, name + '_')
                data['qual'].append(row)

    # write to file
    for key, value in data.items():
        pd.DataFrame(data[key]).to_csv(out_dir / f'{study_name}-{key}Metrics.csv', index=False)
                    

In [9]:
df_bone = pd.read_csv(f'outputs/study1/{study_name}-boneMetrics.csv')
df_cart = pd.read_csv(f'outputs/study1/{study_name}-cartMetrics.csv')
df_qual = pd.read_csv(f'outputs/study1/{study_name}-qualMetrics.csv')

In [13]:
df_bone

,sub,bone,run_id,orig_rmsdi,orig_di_mean,orig_di_median,orig_di_std,orig_di_min,orig_di_max,orig_di_99,...,remesh2d_di_95,remesh2d_rmsdo,remesh2d_do_mean,remesh2d_do_median,remesh2d_do_std,remesh2d_do_min,remesh2d_do_max,remesh2d_do_99,remesh2d_do_95,remesh3d_vol
0,14548R,tpm,0c,0.027484,0.021702,0.018275,0.016863,8.931215e-08,0.129691,0.072749,...,0.002771,0.078397,0.059717,0.048959,0.050794,1.743523e-08,0.283899,0.196294,0.158641,1409.852092
1,14548R,tpm,1c,0.027517,0.021688,0.018302,0.016936,1.074586e-06,0.126677,0.073469,...,0.002722,0.078608,0.060127,0.049246,0.050636,3.015315e-07,0.296187,0.197197,0.158634,1409.647886
2,14548R,tpm,2c,0.027265,0.021489,0.018157,0.016781,8.352395e-07,0.135526,0.072994,...,0.002651,0.079829,0.061108,0.050197,0.051366,3.540940e-08,0.280966,0.199396,0.160504,1409.632893
3,14548R,tpm,3c,0.027497,0.021699,0.018300,0.016888,2.620388e-07,0.135806,0.072729,...,0.002733,0.074886,0.056457,0.045161,0.049199,1.114729e-08,0.302451,0.191771,0.152494,1411.485202
4,14548R,tpm,4c,0.027331,0.021604,0.018353,0.016741,7.171755e-07,0.133374,0.072462,...,0.002769,0.075156,0.056525,0.045197,0.049532,1.605428e-07,0.290943,0.193684,0.154149,1411.430512
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
724,14548R,tpm,724c,0.027396,0.021608,0.018099,0.016841,5.424361e-07,0.135385,0.072954,...,0.002845,0.157317,0.087389,0.033278,0.130812,5.678250e-08,0.906166,0.610985,0.386289,1391.747556
725,14548R,tpm,725c,0.027504,0.021679,0.018261,0.016927,9.011688e-07,0.126431,0.072948,...,0.002823,0.155779,0.086595,0.032777,0.129493,1.434989e-07,0.904702,0.599622,0.381515,1391.747556
726,14548R,tpm,726c,0.027503,0.021671,0.018240,0.016934,3.996951e-07,0.135682,0.072930,...,0.002738,0.129787,0.074379,0.031916,0.106359,5.557645e-08,0.707031,0.522483,0.297835,1399.397922
727,14548R,tpm,727c,0.027491,0.021679,0.018259,0.016905,8.784952e-07,0.133908,0.073614,...,0.002748,0.128296,0.074409,0.031865,0.104514,1.677631e-07,0.715689,0.498114,0.295159,1399.397922


In [11]:
df_cart

,sub,bone,run_id,orig_rmsdi,orig_di_mean,orig_di_median,orig_di_std,orig_di_min,orig_di_max,orig_di_99,...,remesh2d_h_99,remesh2d_h_95,remesh3d_h_rmsd,remesh3d_h_mean,remesh3d_h_median,remesh3d_h_std,remesh3d_h_min,remesh3d_h_max,remesh3d_h_99,remesh3d_h_95
0,14548R,tpm,0c,0.003745,0.002754,0.002057,0.002539,2.334327e-09,0.025976,0.011625,...,0.688387,0.618058,0.452931,0.444180,0.437559,0.088607,0.250769,0.733679,0.682521,0.607463
1,14548R,tpm,1c,0.003728,0.002733,0.002034,0.002535,3.806111e-08,0.028508,0.011772,...,0.694672,0.619897,0.453970,0.444951,0.438812,0.090045,0.250568,0.731410,0.680994,0.610621
2,14548R,tpm,2c,0.003729,0.002744,0.002050,0.002526,1.585042e-08,0.032964,0.011703,...,0.690265,0.619566,0.453410,0.444507,0.437886,0.089410,0.250272,0.735479,0.683004,0.610275
3,14548R,tpm,3c,0.003759,0.002741,0.002031,0.002571,1.761742e-07,0.035431,0.011961,...,0.689779,0.618225,0.453365,0.444416,0.436746,0.089637,0.250567,0.748907,0.684977,0.611832
4,14548R,tpm,4c,0.003742,0.002725,0.002020,0.002564,3.972935e-09,0.033808,0.011845,...,0.690174,0.617376,0.454783,0.445555,0.438341,0.091146,0.251850,0.749595,0.683922,0.614229
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
724,14548R,tpm,724c,0.003773,0.002760,0.002044,0.002572,7.815558e-08,0.032210,0.011819,...,0.692519,0.620680,0.453170,0.444131,0.437733,0.090062,0.251492,0.738471,0.680611,0.610426
725,14548R,tpm,725c,0.003732,0.002737,0.002052,0.002536,6.679812e-08,0.030040,0.011697,...,0.688330,0.617350,0.453143,0.444188,0.436454,0.089642,0.250578,0.740344,0.679410,0.609601
726,14548R,tpm,726c,0.003729,0.002735,0.002047,0.002534,1.855344e-07,0.028386,0.011737,...,0.687555,0.616378,0.454822,0.445758,0.438795,0.090347,0.251828,0.755459,0.682369,0.611561
727,14548R,tpm,727c,0.003719,0.002735,0.002043,0.002520,7.983021e-08,0.029368,0.011682,...,0.689946,0.619580,0.453041,0.444104,0.437160,0.089541,0.252748,0.750833,0.679256,0.610107


In [12]:
df_qual

,sub,bone,run_id,bone_n_cells,bone_min_angle_n_nan,bone_min_angle_mean,bone_min_angle_median,bone_min_angle_std,bone_min_angle_best,bone_min_angle_worst,...,cart_inner_scaled_jacobian_median,cart_inner_scaled_jacobian_std,cart_inner_scaled_jacobian_best,cart_inner_scaled_jacobian_worst,cart_inner_scaled_jacobian_preferred_range_pct,cart_inner_scaled_jacobian_acceptable_range_pct,cart_inner_scaled_jacobian_bad_cells,cart_inner_scaled_jacobian_95%,cart_inner_scaled_jacobian_99%,cart_inner_scaled_jacobian_99.9%
0,14548R,tpm,0c,19547,0,42.126072,45.089336,16.498264,70.518659,0.040072,...,0.631836,0.149958,0.982342,0.002808,91.704845,98.452735,922,0.333092,0.157021,0.030836
1,14548R,tpm,1c,15051,0,41.897288,45.445729,17.111539,70.549690,0.040072,...,0.631702,0.150644,0.982342,0.002808,91.646262,98.389964,960,0.331478,0.150858,0.024392
2,14548R,tpm,2c,14115,0,39.522550,43.496973,17.932192,70.500201,0.040072,...,0.631567,0.151711,0.982342,0.002121,91.556092,98.292789,1019,0.329129,0.143950,0.016959
3,14548R,tpm,3c,19684,0,42.174213,45.090248,16.357627,70.523098,0.066327,...,0.630493,0.149535,0.982342,0.002808,91.656892,98.466772,915,0.334453,0.158459,0.031800
4,14548R,tpm,4c,15260,0,42.150695,45.480725,16.669738,70.530730,0.066327,...,0.630419,0.150055,0.982342,0.002808,91.612374,98.418946,944,0.333166,0.154428,0.025407
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
724,14548R,tpm,724c,62930,0,47.067502,48.244837,13.111400,70.529830,0.196439,...,0.630821,0.149178,0.982342,0.002808,91.753889,98.502839,894,0.334952,0.161084,0.040114
725,14548R,tpm,725c,62930,0,47.067502,48.244837,13.111400,70.529830,0.196439,...,0.630821,0.149178,0.982342,0.002808,91.753889,98.502839,894,0.334952,0.161084,0.040114
726,14548R,tpm,726c,63576,0,47.196954,48.486231,13.130342,70.530841,0.040211,...,0.631213,0.149244,0.985040,0.005618,91.815118,98.479239,916,0.338156,0.163947,0.037523
727,14548R,tpm,727c,63576,0,47.196954,48.486231,13.130342,70.530841,0.040211,...,0.631213,0.149244,0.985040,0.005618,91.815118,98.479239,916,0.338156,0.163947,0.037523


## Runtimes

In [ ]:
from pathlib import Path
import pandas as pd

# -------- PATHS --------------------------------------------- #
root_dir = Path('../../../../MeshPipeline/outputs/ParamOptimisation/criteria3D') # path to parent of output_root in set_parameters
out_dir = Path('outputs/study1') # path dir to save outputs in

study_prefix = 'study1' # start of dir name of output_root in set_parameters
studies = ['a', 'b', 'c'] # individual study identifier (end of dir name of output_root in set_parameters)

run_dfs = []
for study in studies:
    study_name = study_prefix + study
    study_dir = root_dir / study_name

    runtimes = pd.read_json(study_dir / 'reports/runtimes.jsonl', lines=True)
    runtimes = runtimes[runtimes['step']=='3Dmesh'].copy()
    runtimes['bone'] = runtimes['bones'].apply(lambda x: x.split('-')[0])
    runtimes['run_id'] = runtimes['run_ids'].apply(lambda x: str(x[-1])+study)
    run_dfs.append(runtimes[['subject', 'bone', 'run_id', 'runtime']].copy())
pd.concat(run_dfs).to_csv(out_dir / 'runtimes.csv', index=False)


In [8]:
runtimes = pd.read_csv('outputs/study1/runtimes.csv')
runtimes

,subject,bone,run_id,runtime
0,14548R,tpm,0a,12.927883
1,14548R,tpm,1a,14.353772
2,14548R,tpm,2a,11.393350
3,14548R,tpm,3a,13.403551
4,14548R,tpm,4a,14.793903
...,...,...,...,...
2182,14548R,tpm,724c,12.407320
2183,14548R,tpm,725c,11.637659
2184,14548R,tpm,726c,12.816542
2185,14548R,tpm,727c,12.821653


## Params

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import json


root_dir = Path('../../../../MeshPipeline/outputs/ParamOptimisation/criteria3D') # path to parent of output_root in set_parameters
out_dir = Path('outputs/study1') # path dir to save outputs in

study_prefix = 'study1' # start of dir name of output_root in set_parameters
studies = ['a', 'b', 'c'] # individual study identifier (end of dir name of output_root in set_parameters)

rows = []
for study in studies:
    study_name = study_prefix + study
    study_dir = root_dir / study_name

    params_path = study_dir / 'params/full_params.json'
    with open(params_path, 'r') as f:
        params = json.load(f)

    subs = params['subjects']['subject_sideL']
    bone_pairs = params['subjects']['bone_arbone']

    for sub in subs:
        subject, sideL = sub[:-1], sub[-1]

        for bone_pair in bone_pairs:

            bone, arbone = bone_pair.split('-')
            path_mesh = study_dir / f'meshes/{subject}{sideL}/{bone_pair}'
            run_ids = np.sort([int(p.name.split('-')[-1][:-4]) for p in (path_mesh / '3Dmesh').iterdir() if p.suffix == '.vtu'])

            for run_id in run_ids:

                param3d_path = root_dir / f'{study_name}/params/3Dmesh/{run_id}.json'
                with open(param3d_path, 'r') as f:
                    param3d = json.load(f)

                row = {
                    'sub': sub,
                    'bone': bone,
                    'run_id': str(param3d['run_id'])+study,
                    'h_bone_max': param3d['cgal_params']['sizing_field']['h_bone_max']
                }

                for param, val in param3d['_loop'].items():
                    row[param.split('.')[-1]] = val
                
                rows.append(row)
pd.DataFrame(rows).to_csv(out_dir / 'params.csv', index=False)

In [6]:
param_df = pd.read_csv('outputs/study1/params.csv')
param_df

,sub,bone,run_id,h_bone_max,d0,fd_cart_near,fd_cart_far,fd_bone,facet_angle,cell_radius_edge_ratio
0,14548R,tpm,0a,0.5,2,0.02,0.01,0.2,7.5,3
1,14548R,tpm,1a,0.5,2,0.02,0.01,0.2,7.5,6
2,14548R,tpm,2a,0.5,2,0.02,0.01,0.2,7.5,12
3,14548R,tpm,3a,0.5,2,0.02,0.01,0.2,15.0,3
4,14548R,tpm,4a,0.5,2,0.02,0.01,0.2,15.0,6
...,...,...,...,...,...,...,...,...,...,...
2182,14548R,tpm,724c,2.0,8,0.08,0.04,0.8,15.0,6
2183,14548R,tpm,725c,2.0,8,0.08,0.04,0.8,15.0,12
2184,14548R,tpm,726c,2.0,8,0.08,0.04,0.8,30.0,3
2185,14548R,tpm,727c,2.0,8,0.08,0.04,0.8,30.0,6
